## Setup

In [ ]:
BASE_DIR = "D:/thesis/THS-ST3"
MODEL_DIR = r"D:/thesis/model/sentence-transformer"

In [ ]:
from sklearn.cluster import AffinityPropagation
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import torch

## Load language model

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(f'{MODEL_DIR}/sroberta-nli-1epoch')
model

## Load sense inventory

In [ ]:
sense_inventory = pd.read_pickle(fr'{BASE_DIR}/sense_inventory_demo_3step_70cohfie_lrec_config.pkl')
print(sense_inventory.shape)
display(sense_inventory)

## WSI Eval (PWN Eval)

In [ ]:
def preprocess_validation_set(df_validation_set):
    # Copy dataframe
    df = df_validation_set.copy()

    # Drop rows with 'XX', this indicates that there's no example sentences in Princeton WordNet
    to_drop = df[df['fil_text']=='XX']
    print(f'Dropping senses of these words: {to_drop.fil_word.values.tolist()}')
    df.drop(to_drop.index, inplace=True)

    # Expand
    fil_word = []
    eng_word = []
    backtranslation = []
    synset_id = []
    gloss = []
    eng_text = []
    fil_text = []
    for index, row in df.iterrows():
        eng_text_spliited =  row['eng_text'].split(" | ")
        fil_text_splitted =  row['fil_text'].split(" | ")
        for idx in range(len(fil_text_splitted)): # can be eng_text, basta pantay lang yan
            fil_word.append(row['fil_word'])
            eng_word.append(row['eng_word'])
            backtranslation.append(row['backtranslation'])
            synset_id.append(row['synset_id'])
            gloss.append(row['gloss'])
            eng_text.append(eng_text_spliited[idx])
            fil_text.append(fil_text_splitted[idx])

    return pd.DataFrame({'fil_word': fil_word,
                         'eng_word': eng_word,
                         'backtranslation': backtranslation,
                         'synset_id': synset_id,
                         'gloss': gloss,
                         'eng_text': eng_text,
                         'fil_text': fil_text})
    
def remove_sentences_no_target_word(df):
    '''
    Drop sentences that doesn't contain the target word (excluding its inflected forms)
    '''
    to_drop_idx = [] # list of row indexes to drop

    for idx, row in df.iterrows():
        target_word = f"{row['fil_word']}"
        if target_word not in row['fil_text'].split(" "): # if fil_word not found in sentence
            to_drop_idx.append(idx)
            #print(f"{row['fil_word']} : idx {idx} : {row['fil_text']}")

    print(f"Total rows to drop: {len(to_drop_idx)}")

    return df.drop(to_drop_idx)

def WSD_EVAL(df_text_embs, df_senses, target_word, thres=0.5):
    # Initialize 
    cos = torch.nn.CosineSimilarity(dim=1, eps=1e-08)
    sentence_embeddings = torch.tensor(df_text_embs['embs'].values.tolist())

    lst = []

    senses = df_senses.loc[df_senses['word']==target_word]

    senses_dict = {}

    for row in senses.itertuples():
        senses_dict[row.sense_id] = {'sense_embedding': row.sense_embedding}

    for idx, sense in enumerate(senses_dict):
        output = cos(sentence_embeddings, torch.tensor(senses_dict[sense]['sense_embedding']).unsqueeze(0))
        lst.append(output)

    res = torch.stack(lst, dim=1)
    labels = torch.argmax(res, dim=1).tolist()
    scores = torch.max(res, dim=1)[0].tolist()

    sense_ids = list(senses_dict.keys())

    # if less than threshold, label as 'XX'

    mapped_labels = []
    for idx, score in enumerate(scores):
        if score < thres:
            mapped_labels.append('XX')
        else:
            mapped_labels.append(sense_ids[labels[idx]])

    # mapped_labels = []
    # for label in labels:
    #     mapped_labels.append(sense_ids[label])

    # # Append labels to DataFrame
    # df_text_embs.loc[:, 'sense_id'] = mapped_labels
    # df_text_embs.loc[:, 'score'] = scores

    return mapped_labels, scores

def get_embeddings_validation(model, df_validation_set):
    '''
    Breaks on single examples
    '''
    sentence_list_texts = df_validation_set['fil_text'].values.tolist()
    sentence_embeddings = model.encode(sentence_list_texts)
    df_validation_set['embs'] = list(sentence_embeddings)
    
    return df_validation_set

def validate(df_validation_set, df_sense_inventory, thres=0.5):
    df = df_validation_set.copy()
    df =  get_embeddings_validation(model, df)

    # loop over each unique fil_word
    fil_words = df['fil_word'].unique().tolist()
    for word in fil_words:
        # # WSD, add sense tag and score
        labels, scores = WSD_EVAL(df[df['fil_word']==word], df_sense_inventory, word, thres=thres)
        df.loc[df['fil_word']==word, 'sense_id'] = labels
        df.loc[df['fil_word']==word, 'score'] = scores

    return df

In [ ]:
validation_set = pd.read_csv(f'{BASE_DIR}/full_vocab_evaluation_final.csv') # Load the validation data
validation_set = preprocess_validation_set(validation_set) # Preprocess the validation data
validation_set = remove_sentences_no_target_word(validation_set) # Drop sentences with no target_word
validation_set

In [ ]:
len(validation_set['synset_id'].unique())

## How many words lang ang ma-validate against our own WordNet?

Only 23% ng FilWordNet lang yung pwede ma-validate

In [ ]:
a = len(validation_set['fil_word'].unique())
b = len(sense_inventory['word'].unique())

print(f"Words na ma-validate: {a}")
print(f"Unique words in our wordnet: {b}")
print(f"Percentage of our wordnet na ma-validate: {round((a / b) * 100)}%")

## Get cosine similarity score distribution

In [ ]:
results_temp = validate(validation_set, sense_inventory, thres=0.5)

In [ ]:
score_distribution = results_temp['score'].describe()
score_distribution

## THRES = MEAN

In [ ]:
mean = results_temp['score'].mean()
mean

In [ ]:
results = validate(validation_set, sense_inventory, thres=mean)
results

In [ ]:
validation_words = results['fil_word'].unique().tolist() # some words may be dropped kaya we need to get it again
sense_ids = sense_inventory[sense_inventory['word'].isin(validation_words)].groupby('sense_id').count()['word'] # get all sense_ids for words in validation set
sense_tag_counts = results.groupby(['sense_id']).count()['fil_word'] # get counts of sense tags

# Get the counts
sense_tag_counts = pd.merge(sense_ids,sense_tag_counts,on='sense_id',how='left').fillna(0)
sense_tag_counts.rename(columns={'fil_word': 'count'}, inplace=True)
sense_tag_counts.drop(['word'], axis=1, inplace=True)
sense_tag_counts['word'] = [s[:-2] for s in sense_tag_counts.index.tolist()]
sense_tag_counts

In [ ]:
# How many are valid
valid = sense_tag_counts.loc[sense_tag_counts['count'] >= 1.0]
valid

In [ ]:
# ilang words ang may at least 1 sense yung na-validate?
len(valid['word'].unique())

In [ ]:
valid_counts = valid.shape[0]
valid_counts

In [ ]:
total_sense_count = sense_tag_counts.shape[0]
total_sense_count

In [ ]:
(valid_counts/total_sense_count)*100

In [ ]:
sense_tag_counts.plot.hist()

## THRES = MEAN + 1 STD

In [ ]:
mean_1std = score_distribution['mean'] + score_distribution['std']
mean_1std

In [ ]:
results = validate(validation_set, sense_inventory, thres=mean_1std)
results

In [ ]:
validation_words = results['fil_word'].unique().tolist() # some words may be dropped kaya we need to get it again
sense_ids = sense_inventory[sense_inventory['word'].isin(validation_words)].groupby('sense_id').count()['word'] # get all sense_ids for words in validation set
sense_tag_counts = results.groupby(['sense_id']).count()['fil_word'] # get counts of sense tags

# Get the counts
sense_tag_counts = pd.merge(sense_ids,sense_tag_counts,on='sense_id',how='left').fillna(0)
sense_tag_counts.rename(columns={'fil_word': 'count'}, inplace=True)
sense_tag_counts.drop(['word'], axis=1, inplace=True)
sense_tag_counts['word'] = [s[:-2] for s in sense_tag_counts.index.tolist()]
sense_tag_counts

In [ ]:
# How many are valid
valid = sense_tag_counts.loc[sense_tag_counts['count'] >= 1.0]
valid

In [ ]:
# ilang words ang may at least 1 sense yung na-validate?
len(valid['word'].unique())

In [ ]:
valid_counts = valid.shape[0]
valid_counts

In [ ]:
total_sense_count = sense_tag_counts.shape[0]
total_sense_count

In [ ]:
(valid_counts/total_sense_count)*100

In [ ]:
sense_tag_counts.plot.hist()

## THRES = MEAN - 1 STD

In [ ]:
mean_1std = score_distribution['mean'] - score_distribution['std']
mean_1std

In [ ]:
results = validate(validation_set, sense_inventory, thres=mean_1std)
results

In [ ]:
validation_words = results['fil_word'].unique().tolist() # some words may be dropped kaya we need to get it again
sense_ids = sense_inventory[sense_inventory['word'].isin(validation_words)].groupby('sense_id').count()['word'] # get all sense_ids for words in validation set
sense_tag_counts = results.groupby(['sense_id']).count()['fil_word'] # get counts of sense tags

# Get the counts
sense_tag_counts = pd.merge(sense_ids,sense_tag_counts,on='sense_id',how='left').fillna(0)
sense_tag_counts.rename(columns={'fil_word': 'count'}, inplace=True)
sense_tag_counts.drop(['word'], axis=1, inplace=True)
sense_tag_counts

In [ ]:
# How many are valid
sense_tag_counts.loc[sense_tag_counts['count'] >= 1.0]

In [ ]:
valid_counts = sense_tag_counts.loc[sense_tag_counts['count'] >= 1.0].shape[0]
valid_counts

In [ ]:
total_sense_count = sense_tag_counts.shape[0]
total_sense_count

## THRES = 0

In [ ]:
results = validate(validation_set, sense_inventory, thres=-1)
results

In [ ]:
validation_words = results['fil_word'].unique().tolist() # some words may be dropped kaya we need to get it again
sense_ids = sense_inventory[sense_inventory['word'].isin(validation_words)].groupby('sense_id').count()['word'] # get all sense_ids for words in validation set
sense_tag_counts = results.groupby(['sense_id']).count()['fil_word'] # get counts of sense tags

# Get the counts
sense_tag_counts = pd.merge(sense_ids,sense_tag_counts,on='sense_id',how='left').fillna(0)
sense_tag_counts.rename(columns={'fil_word': 'count'}, inplace=True)
sense_tag_counts.drop(['word'], axis=1, inplace=True)
sense_tag_counts

In [ ]:
sense_tag_counts.loc[sense_tag_counts['count'] < 1.0]

In [ ]:
# How many are valid
sense_tag_counts.loc[sense_tag_counts['count'] >= 1.0]

In [ ]:
valid_counts = sense_tag_counts.loc[sense_tag_counts['count'] >= 1.0].shape[0]
valid_counts

In [ ]:
total_sense_count = sense_tag_counts.shape[0]
total_sense_count

In [ ]:
(valid_counts/total_sense_count)*100

In [ ]:
(valid_counts/total_sense_count)*100

### Sense Consistency

In [ ]:
sns.set(font_scale=1.1)

In [ ]:
import seaborn as sns
import matplotlib.pylab as plt

for word in validation_words:
    df_grouped = results[results['fil_word']==word].groupby(['synset_id', 'sense_id'], as_index=False)
    eng_word = results[results['fil_word']==word]['eng_word'].unique()[0] # get english word for plot title
    print(eng_word)
    freq = df_grouped.size()

    freq['normalized_size'] = 0         # intialize

    for index,row in freq.iterrows():
        n_instances = freq[freq['synset_id']==row.synset_id]['size'].sum()
        count = row['size']
        freq.loc[index, 'normalized_size'] = count / n_instances

    #freq.drop(freq[freq['sense_id'] == 'XX'].index, inplace=True)
    pivot = freq.pivot(index='synset_id', columns='sense_id', values='normalized_size')
    pivot.fillna(0, inplace=True)

    # Plot heatmap
    plt.figure(figsize=(8,5))
    hm = sns.heatmap(pivot.transpose(), linewidth=1, linecolor='w', square=True, annot=True, cbar=False)
    hm.set_xlabel("PWN sense")
    hm.set_ylabel("Our sense")
    plt.title(f'target word: {word} ({eng_word})')
    plt.yticks(rotation=0)
    plt.xticks(rotation=0)
    #plt.savefig(f'{BASE_DIR}/plots/{word}.png') # save to disk
    plt.savefig(f'{BASE_DIR}/plots/{word}.pdf') # PDF lossless
    plt.show()

## Synset Induction and Eval

### Synset Induction

In [ ]:
import torch
import numpy as np
import pandas as pd
import random
from sklearn import preprocessing
from sklearn.cluster import KMeans
from sklearn.cluster import AffinityPropagation
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_samples, silhouette_score

def generate_synsets(sense_inventory, n_clusters=None, mode="KMEANS", random_state=0, distance_threshold=0.12):
    '''
        df_sense_embeddings: columns=['word', 'sense_id', 'example_sentences', 'pos', 'contextual_info', 'sense_embedding']
    '''
    df_synsets = sense_inventory.copy()
    sense_embeddings = list(sense_inventory['sense_embedding'])
    
    af = AgglomerativeClustering(n_clusters=None, 
                                     affinity="cosine", 
                                     linkage="average", 
                                     distance_threshold=distance_threshold,
                                     compute_distances=True).fit(sense_embeddings)

    df_synsets['synset_id'] = af.labels_

    return df_synsets

### Eval

In [ ]:
def get_jaccard_index(lst_a, lst_b):
    intersection = list(set(lst_a).intersection(lst_b))
    union = list(set(lst_a).union(lst_b))
    return len(intersection)/len(union)
    
def evaluate_synsets(our_synsets, fwn_synsets, fwn_synsets_with_definition):
    '''
    DATAFRAME VERSION
    Returns DataFrame with the ff. columns: our_synset_index, fwn_synset_index, our_synsets, fwn_synsets, jaccard_scores
    Does not evaluate synsets with single item.
    '''
    ##############################
    # BUILD JACCARD INDEX MATRIX
    ##############################
    jaccard_matrix = [] # store 
    #invalid_synset_idx = []
    for idx, our_synset in enumerate(our_synsets):
        if len(our_synset) == 1:
            pass
            #invalid_synset_idx.append(idx)

        score = []
        for fwn_synset in fwn_synsets:
            #score.append(get_jaccard_index(our_synset, fwn_synset))
            if len(fwn_synset) == 1: # if fwn synset is just 1, make the score -1 para never ma select sa argmax
                score.append(-1)
            else:
                score.append(get_jaccard_index(our_synset, fwn_synset))
        
        jaccard_matrix.append(score)

    ##############################
    # GET HIGHEST JACCARD INDEX
    ##############################
    jaccard_matrix = np.array(jaccard_matrix)

    ##############################
    # BUILD RESULTS (our_synset_index, fwn_synset_index, jaccard_scores)
    # REMOVE INVALID SYNSETS (SINGLE ITEM SYNSETS)
    ##############################
    our_synset_index = np.array(range(len(our_synsets)))
    #our_synset_index = np.delete(our_synset_index, invalid_synset_idx)
    fwn_synset_index = np.argmax(jaccard_matrix, axis=1)
    #fwn_synset_index = np.delete(fwn_synset_index, invalid_synset_idx)

    our_synsets = np.array(our_synsets_with_sense_id)[our_synset_index]
    fwn_synsets = np.array(fwn_synsets)[fwn_synset_index]
    fwn_synsets_with_definition = np.array(fwn_synsets_with_definition)[fwn_synset_index]

    jaccard_scores = np.round(jaccard_matrix[our_synset_index, fwn_synset_index], decimals=2)

    #return np.column_stack((our_synsets, fwn_synsets, jaccard_scores))
    df_results = pd.DataFrame(data={'our_synset_index': our_synset_index, 'fwn_synset_index': fwn_synset_index, 'our_synsets': our_synsets, 'fwn_synsets': fwn_synsets, 'fwn_definition': fwn_synsets_with_definition, 'jaccard_scores': jaccard_scores}).sort_values(by=['jaccard_scores'], ascending=False).reset_index(drop=True)

    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', 150):
        display(df_results[['our_synsets', 'fwn_synsets', 'fwn_definition', 'jaccard_scores']])

    return df_results

def display_synset(synset):
    for sense_id in synset:
        example_sentences = sense_inventory[sense_inventory['sense_id'] == sense_id].example_sentences.values
        print(sense_id)
        for sent in example_sentences[0]:
            print(f"   {sent}")

In [ ]:
%%time

dist_thresh = [0.015, 0.018, 0.02, 0.025, 0.05, 0.075, 0.1, 0.12, 0.15, 0.17, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45]
dist_thresh = [0.05, 0.075, 0.1, 0.12, 0.15]
# dist_thresh = [0.1, 0.12]

df_synset = []

for thresh in dist_thresh:
  df_synset.append(generate_synsets(sense_inventory, distance_threshold = thresh))

display(df_synset)

In [ ]:
len(df_synset[13]['synset_id'].unique())

In [ ]:
df_synset[0]

### Preprocess for Eval

In [ ]:
# Load FWN
df_fwn = pd.read_csv(f"{BASE_DIR}/filwordnet.csv") # Load Filipino WordNet

# Precoess FilWordNet synsets for evaluation
fwn_synsets = []
fwn_synsets_with_definition = []
for synsetid in df_fwn['synsetid'].unique():
    synset_item = list(df_fwn[df_fwn['synsetid'] == synsetid]['lemma'])
    fwn_synsets.append(synset_item)
    synset_item_definition =  list(df_fwn[df_fwn['synsetid'] == synsetid]['definition']) 
    fwn_synsets_with_definition.append(synset_item_definition)
  
# Precoess our synsets for evaluation
# our_synsets = []
# our_synsets_with_sense_id = []
# i = 1
# for synset_id in df_synset[i]['synset_id'].unique():
#     synset_item = list(df_synset[i][df_synset[i]['synset_id'] == synset_id]['word'])
#     our_synsets.append(synset_item)
#     synset_item_sense_id = list(df_synset[i][df_synset[i]['synset_id'] == synset_id]['sense_id'])
#     our_synsets_with_sense_id.append(synset_item_sense_id)
# df_results = evaluate_synsets(our_synsets, fwn_synsets, fwn_synsets_with_definition)
# df_results['jaccard_scores'].mean()

# Evaluate Jaccard Scores
df_scores = []
count = 0
for df in df_synset[1:]:
  our_synsets = []
  our_synsets_with_sense_id = []
  count = count+1
  print(count)
  for synset_id in df['synset_id'].unique():
      synset_item = list(df[df['synset_id'] == synset_id]['word'])
      our_synsets.append(synset_item)
      synset_item_sense_id = list(df[df['synset_id'] == synset_id]['sense_id'])
      our_synsets_with_sense_id.append(synset_item_sense_id)
      
  df_results = evaluate_synsets(our_synsets, fwn_synsets, fwn_synsets_with_definition)
  df_scores.append(df_results[df_results['jaccard_scores']>0.6]['jaccard_scores'].count())


In [ ]:
for i in range(len(df_scores)):
  print(str(dist_thresh[i]) + ': ' + str(df_scores[i]))

In [ ]:
for i in range(len(df_scores)):
  print(str(dist_thresh[i]) + ': ' + str(df_scores[i]))

In [ ]:
df_results = evaluate_synsets(our_synsets, fwn_synsets, fwn_synsets_with_definition)

In [ ]:
df_results['jaccard_scores'].mean()

In [ ]:
df_results_filtered = df_results.loc[df_results['our_synsets'].str.len() > 1].reset_index()
df_results_filtered = df_results_filtered[['our_synsets', 'fwn_synsets', 'fwn_definition', 'jaccard_scores']]

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', 150):
    display(df_results_filtered)

In [ ]:
fig, ax = plt.subplots()
df_results_filtered['jaccard_scores'].value_counts().sort_index().plot(ax=ax, kind='bar')
ax.tick_params(axis='x', rotation=90)
ax.set_xlabel("Jaccard similarity")
ax.set_ylabel("Frequency")
#df_results['jaccard_scores'].plot.hist(bins=10)

In [ ]:
df_results_filtered['jaccard_scores'].value_counts().sort_index()

In [ ]:
(df_synset['synset_id'].value_counts()).value_counts().sort_index()

In [ ]:
 df_synset['synset_id'].value_counts().describe()